# Threshold calibration Drive workflow

该 notebook 用于在 Colab 中从 Google Drive 前序结果包生成 threshold calibration 结果包, 并把 zip 保存回 Google Drive。


In [ ]:
SLM_WM_PAPER_RUN_NAME = "pilot_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
# 依赖诊断已收敛到 paper_workflow.colab_utils.dependency_check, 在仓库 checkout 后统一执行。


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="threshold_calibration",
)
print(paper_run_environment)

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report
dependency_report = build_notebook_dependency_report("threshold_calibration")
print(dependency_report)


In [ ]:
import os

from paper_workflow.colab_utils.threshold_calibration import run_default_threshold_calibration_from_drive_plan

threshold_calibration_summary = run_default_threshold_calibration_from_drive_plan(
    root='.',
    attention_injection_drive_dir=os.environ['SLM_WM_ATTENTION_INJECTION_DRIVE_DIR'],
    aligned_rescoring_drive_dir=os.environ['SLM_WM_ALIGNED_RESCORING_DRIVE_DIR'],
    target_fpr=float(os.environ['SLM_WM_THRESHOLD_TARGET_FPR']),
    max_content_records=os.environ['SLM_WM_THRESHOLD_MAX_CONTENT_RECORDS'],
    minimum_clean_negative_count=os.environ['SLM_WM_THRESHOLD_MINIMUM_CLEAN_NEGATIVE_COUNT'],
)
threshold_calibration_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/threshold_calibration/threshold_calibration_result.json')
input_manifest_path = Path('outputs/threshold_calibration/threshold_calibration_input_package_manifest.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else threshold_calibration_summary)
if input_manifest_path.exists():
    print(input_manifest_path.read_text(encoding='utf-8'))


In [ ]:
from paper_workflow.colab_utils.notebook_entrypoint import package_workflow_outputs

archive_record = package_workflow_outputs(root='.', workflow_name="threshold_calibration")
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/threshold_calibration').glob('*.json')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/threshold_calibration').glob('*.csv')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

# 该 Notebook 的职责是完成受治理产物落盘和打包。
# threshold_calibration_ready 表示 fixed-FPR 效果门禁是否通过; 真实分数校准后, 方法效果失败也应保留完整结果包供审计。
assert threshold_calibration_summary['geometric_rescue_ready'] is True, threshold_calibration_summary
assert threshold_calibration_summary['metadata']['score_space_alignment_ready'] is True, threshold_calibration_summary
assert threshold_calibration_summary['metadata']['real_score_calibration_ready'] is True, threshold_calibration_summary
assert threshold_calibration_summary['metadata']['proxy_score_calibration_used'] is False, threshold_calibration_summary
assert archive_record.archive_digest == archive_record.drive_archive_digest, archive_record.to_dict()

if not threshold_calibration_summary['threshold_calibration_ready']:
    print('fixed-FPR 协议未通过, 这表示真实分数空间下方法效果未达标; 结果包仍已落盘, 后续应修复真实 latent content carrier。')

threshold_calibration_summary
